# pyfixest + MLflow tracking example

This notebook makes some synthetic data, runs it through `run_experiment` (which wraps a pyfixest modeling function and logs to MLflow), and then pulls the logged runs back out to inspect them.

In [1]:
import mlflow
import numpy as np
import pandas as pd

from tracking import run_experiment

mlflow.set_tracking_uri("sqlite:///mlflow.db")
# Set the experiment once. run_experiment then logs into the active experiment,
# so you don't pass experiment_name on every call.
mlflow.set_experiment("pyfixest-example")

/home/user/experiment-templates/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026/07/16 17:50:42 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/07/16 17:50:42 INFO mlflow.store.db.utils: Updating database tables


2026/07/16 17:50:44 INFO mlflow.tracking.fluent: Experiment with name 'pyfixest-example' does not exist. Creating a new experiment.


<Experiment: artifact_location='/home/user/experiment-templates/pyfixest-regression/mlruns/1', creation_time=1784224244271, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1784224244271, lifecycle_stage='active', name='pyfixest-example', tags={}, trace_location=None, workspace='default'>

## Make some data

A simple linear DGP: `Y = 2 + 1.5*X1 - 0.8*X2 + noise`.

In [2]:
rng = np.random.default_rng(0)
n = 500

X1 = rng.normal(size=n)
X2 = rng.normal(size=n)
Y = 2.0 + 1.5 * X1 - 0.8 * X2 + rng.normal(scale=1.0, size=n)

data = pd.DataFrame({"Y": Y, "X1": X1, "X2": X2})
data.head()

,Y,X1,X2
0,2.338183,0.125730,1.292893
1,1.742733,-0.132105,0.453671
2,4.504816,0.640423,-1.690160
3,3.005850,0.104900,-0.728195
4,-1.155505,-0.535669,1.232303


## Run it

Two runs on the same data: one with `iid` errors, one with `hetero` (heteroskedasticity-robust) errors. The `vcov` is part of each run's content hash, so these are two distinct runs rather than one deduplicated run. Neither call passes `experiment_name` — they log into the experiment set above.

In [3]:
fit_iid = run_experiment("Y ~ X1 + X2", data=data, vcov="iid")
fit_hetero = run_experiment("Y ~ X1 + X2", data=data, vcov="hetero")

fit_iid.summary()

###

Estimation:  OLS
Dep. var.: Y
sample: None = all
Inference:  iid
Observations:  500

| Coefficient   |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:--------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| Intercept     |      2.049 |        0.045 |    45.121 |      0.000 |  1.960 |   2.139 |
| X1            |      1.525 |        0.045 |    34.144 |      0.000 |  1.438 |   1.613 |
| X2            |     -0.827 |        0.048 |   -17.128 |      0.000 | -0.922 |  -0.732 |
---
RMSE: 1.01 R2: 0.745 


## Get the runs data and show it

`run_experiment` returns the fitted model directly, but everything it logged is also in MLflow. `mlflow.search_runs` pulls that back as a DataFrame.

In [4]:
runs = mlflow.search_runs(experiment_names=["pyfixest-example"])
runs[
    [
        "run_id",
        "params.fml",
        "params.vcov",
        "metrics.r2",
        "metrics.f_statistic",
        "metrics.nobs",
    ]
]

,run_id,params.fml,params.vcov,metrics.r2,metrics.f_statistic,metrics.nobs
0,adeed0cb76a841b7b030f3b449bc4ef5,Y ~ X1 + X2,hetero,0.74513,1128.703709,500.0
1,caae4b0ed1364ebdb3c479c7a177f24e,Y ~ X1 + X2,iid,0.74513,1177.942722,500.0


## Artifacts logged per run

Each run also logs the coefficient table (`coefficients.json`) and a human-readable regression table (`summary.html`, rendered with pyfixest's `etable`).

In [5]:
run_id = runs.loc[0, "run_id"]
[a.path for a in mlflow.artifacts.list_artifacts(run_id=run_id)]

['coefficients.json', 'summary.html']